# Milestone - Unit 7: speed planning

Unit 7 asked how fast the car can go if it **plans** the lap's speed instead of reacting to the corner ahead, and how hard that plan may push.
`agents/unit-07.py` is Unit 6's file with `SPEED_MODE = "profile"` (the default): a forward–backward speed profile planned once in `__init__` (each waypoint's cap √(A_LAT·GRIP[section] / |κ_i|) at most 45 m/s, a backward braking pass at `A_DEC`, a forward pass at `A_ACC`, each twice round the lap), followed with a lead of `PROFILE_LEAD_S` = 0.3 s of travel.
Pure pursuit's lookahead is `PP_K` = 0.5 s; the planner's rates are A_LAT = 4.6 m/s² and A_DEC = 5.0 m/s², one grid step inside the clean edge; the six per-section knobs `GRIP` ship at 1.0, because they are per track.

The notebook asserts the unit's gates (plan `roar-006`) on the trimmed subset the plan names; a routed racekit test (`tests/test_racekit_unit06_07_premise.py`) races every multi-seed figure in full:

- **U7.1** the lookahead at 45 m/s: `PP_K = 0.3` collides (seed 0: 10 collisions), and the reference's 0.5 is clean (U7.4's races);
- **U7.2** the lead at the edge (A_DEC 4.0, A_LAT 5.0), seed 0: a 0.15 s lead collides (6 collisions), 0.3 s is clean, 303.25 s, max |offset| 1.55 m;
- **U7.4** the reference on `test-circuit`, seeds 0–4: all clean, median 304.15 s; `check_submission` passes; its section map is racekit's `grip_sections` (U7.9);
- **U7.6** the stacking trap: the single-knob results stacked, (1.2, 1.5, 1.1, 1.1, 1.05, 1.1), collide 23 times on seed 0;
- **U7.7** the accepted knobs (1.15, 1.3, 1.0, 1.0, 1.0, 1.0), seeds 0–4: all clean, median 299.60 s, faster than the reference beyond the noise.

Every race is on `test-circuit` with racekit's carla-like car, `carla_like(noise=True)`, seeded by `run_race`, `max_seconds=600`.
Tolerances (the plan's Gates header): race times and medians within one tick (±0.05 s); max |offset| within ±0.01 m; outcomes and collision counts exact; noise-aware comparisons strict.
A race is **clean** when it finishes with 0 collisions and its max |cross-track error| on driving rows is at most 2 m.
Every figure is a model number on racekit's carla-like car, not CARLA's.

## 1. Set-up

The milestone runs from `projects/racekit/milestones/`, so the agents are one directory up.
A variant is the reference file with some constants changed (`write_variant`), written to a temporary directory.

In [ ]:
from pathlib import Path
from tempfile import mkdtemp

import numpy as np
from racekit import write_variant
from racekit.cars import carla_like
from racekit.check_submission import check_submission
from racekit.measure import noise_aware_better, repeat_races
from racekit.runner import load_submission, run_race, shim_installed
from racekit.tracks import grip_sections, load_track

AGENTS = Path("..") / "agents"
AGENT = AGENTS / "unit-07.py"
SEEDS = (0, 1, 2, 3, 4)
TICK = 0.05 + 1e-9
OFFSET_TOL = 0.01 + 1e-9
CLEAN_OFFSET = 2.0  # m
workdir = Path(mkdtemp(prefix="roar-milestone-07-"))


def variant(name, **constants):
    return write_variant(AGENT, workdir / f"{name}.py", {key: repr(value) for key, value in constants.items()})


def max_offset(result):
    telemetry = result.telemetry
    return float(np.max(np.abs(telemetry["offset"][telemetry["held"] == 0])))


def clean(result):
    return result.finished and result.collisions == 0 and max_offset(result) <= CLEAN_OFFSET


def race(source, seed=0):
    return run_race(source, "test-circuit", car=carla_like(noise=True), seed=seed, max_seconds=600.0)


def seeds_race(source):
    return repeat_races(source, "test-circuit", n=len(SEEDS), seeds=SEEDS, car_factory=lambda: carla_like(noise=True),
                        max_seconds=600.0)

## 2. U7.4: the reference, and the submission checks

Five seeds on `test-circuit`.
Then `check_submission`, and the section map: the agent's section of every waypoint must be racekit's `grip_sections` on all three tracks (the knobs are only meaningful if they cut the lap where the grip sections do).

In [ ]:
reference = seeds_race(AGENT)
print("reference, seeds 0-4:", [f"{time:.2f}" for time in reference.times],
      f"median {reference.median:.2f} s (spread {reference.spread:.2f})")
print("max |offset| per seed:", [round(max_offset(result), 2) for result in reference.results])
assert all(clean(result) for result in reference.results)
assert abs(reference.median - 304.15) <= TICK

report = check_submission(AGENT)
print(report)
assert report.ok, report.findings

with shim_installed():
    Agent, _ = load_submission(AGENT)
for name in ("oval", "test-circuit", "monza-like"):
    track = load_track(name)
    assert Agent(track.waypoints, None).section == grip_sections(track).tolist(), name
print("the agent's section map is racekit's grip_sections on all three tracks")

## 3. U7.1 and U7.2: looking further ahead at speed

Unit 5's lookahead, `PP_K = 0.3`, on seed 0; the reference's 0.5 is the five clean races above.
Then the lead at the grid's edge, A_DEC 4.0 and A_LAT 5.0, with 0.15 s and with the reference's 0.3 s.

In [ ]:
short_lookahead = race(variant("pp-k-0.3", PP_K=0.3))
print(f"PP_K = 0.3, seed 0: {short_lookahead.collisions} collisions, finished {short_lookahead.finished}")
assert short_lookahead.collisions == 10 and not short_lookahead.finished
assert all(clean(result) for result in reference.results)  # PP_K = 0.5

short_lead = race(variant("edge-lead-0.15", A_DEC=4.0, A_LAT=5.0, PROFILE_LEAD_S=0.15))
lead = race(variant("edge-lead-0.3", A_DEC=4.0, A_LAT=5.0))
print(f"(A_DEC 4.0, A_LAT 5.0), seed 0: lead 0.15 s {short_lead.collisions} collisions; "
      f"lead 0.3 s {lead.total_time:.2f} s, max |offset| {max_offset(lead):.2f} m")
assert short_lead.collisions == 6 and not clean(short_lead)
assert clean(lead) and abs(lead.total_time - 303.25) <= TICK and abs(max_offset(lead) - 1.55) <= OFFSET_TOL

## 4. U7.6 and U7.7: per-section knobs

The single-knob results stacked collide (seed 0; the routed test races seeds 0–9).
The accepted vector of the sequential pass, one step back on the raised knobs, is clean on seeds 0–4 and beats the reference beyond the noise (the routed test checks it on seeds 0–9, and against the fastest clean global setting).

In [ ]:
stacked = race(variant("stack", GRIP=(1.2, 1.5, 1.1, 1.1, 1.05, 1.1)))
print(f"stacked single-knob results, seed 0: {stacked.collisions} collisions, finished {stacked.finished}")
assert stacked.collisions == 23 and not clean(stacked)

accepted = seeds_race(variant("accepted", GRIP=(1.15, 1.3, 1.0, 1.0, 1.0, 1.0)))
print("accepted knobs, seeds 0-4:", [f"{time:.2f}" for time in accepted.times],
      f"median {accepted.median:.2f} s (spread {accepted.spread:.2f})")
assert all(clean(result) for result in accepted.results)
assert abs(accepted.median - 299.60) <= TICK
assert noise_aware_better(accepted.times, reference.times)
print(f"the knobs beat the reference beyond the noise: {reference.median - accepted.median:.2f} s "
      f"> {max(reference.spread, accepted.spread):.2f} s")

## Unit 7, pinned

| on `test-circuit`, carla-like car | median, seeds 0–4 | |
|---|---|---|
| Unit 6's cap (U6.4) | 422.65 s | clean, never above about 20 m/s |
| **Unit 7's profile, (A_DEC 5.0, A_LAT 4.6)** | **304.15 s** | clean, one grid step inside the edge |
| with the accepted knobs (1.15, 1.3, 1, 1, 1, 1) | 299.60 s | clean, faster beyond the noise |

The submission ships the knobs at 1.0: a section is a sixth of *this* lap, so the knobs are per track.
The Unit 7 lab races the shipped agent on CARLA, as it is, and tunes the knobs there.